# Marketplace Intelligence — Product Analytics

**Цель:** исследовать продуктовые метрики на основе PostgreSQL Data Marts: KPI, conversion, retention, behavioral segments и temporal activity.

Ноутбук загружает готовые витрины из схемы `analytics`; SQL остаётся единым источником определений метрик.

### Ограничения
- Нет надёжных price/currency полей: GMV, Revenue и AOV не рассчитываем.
- Funnel — baseline user reach за весь период, не последовательная и не session-level воронка.
- Activity segments измеряют число событий, а не customer value.
- Retention рассчитан по календарным неделям от первого наблюдаемого события; последние когорты right-censored.
- Временные метки интерпретируются как UTC.
- Связь item → category не подтверждена, поэтому category analysis исключён.

In [ ]:
# Imports and project setup
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database import engine

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# Connection check
with engine.connect() as connection:
    database_version = connection.execute(text("SELECT version()")).scalar()
    materialized_views = connection.execute(text("""
        SELECT matviewname
        FROM pg_matviews
        WHERE schemaname = 'analytics'
        ORDER BY matviewname
    """)).scalars().all()

print(database_version.split(",")[0])
print("Materialized views:", materialized_views)


## 1. Load Data Marts

Перед запуском выполни SQL-файлы из `sql/marts/` в числовом порядке. Если витрины не созданы или не обновлены после загрузки данных, запросы ниже завершатся ошибкой либо покажут устаревший snapshot.

In [ ]:
# Load trusted analytics marts from PostgreSQL
def load_mart(table_name: str) -> pd.DataFrame:
    """Load a materialized view from the analytics schema."""
    query = text(f"SELECT * FROM analytics.{table_name}")
    with engine.connect() as connection:
        return pd.read_sql(query, connection)

daily = load_mart("dm_daily_kpis")
funnel = load_mart("dm_conversion_funnel")
retention = load_mart("dm_weekly_retention")
segments = load_mart("dm_user_segments")
temporal = load_mart("dm_temporal_activity")

daily["event_date"] = pd.to_datetime(daily["event_date"])
retention["cohort_week"] = pd.to_datetime(retention["cohort_week"])

for name, frame in {
    "daily": daily, "funnel": funnel, "retention": retention,
    "segments": segments, "temporal": temporal,
}.items():
    print(f"{name:>12}: {len(frame):>10,} rows")


## 2. Data quality and reconciliation

Проверяем зерно каждой витрины, event decomposition, корректность W0 и совпадение общего числа событий между дневной и пользовательской витринами.

In [ ]:
# Data quality checks
for frame in [daily, funnel, retention, segments, temporal]:
    assert not frame.empty, "A required analytics mart is empty"

assert daily["event_date"].is_unique
assert not segments["visitorid"].duplicated().any()
assert not retention.duplicated(["cohort_week", "week_number"]).any()
assert not temporal.duplicated(["day_of_week", "hour_of_day"]).any()

assert (daily["total_events"] == daily["views"] + daily["cart_events"] + daily["purchase_events"]).all()
assert (temporal["total_events"] == temporal["views"] + temporal["add_to_carts"] + temporal["purchases"]).all()
assert (segments["total_events"] == segments["views"] + segments["add_to_carts"] + segments["purchases"]).all()

assert daily["dau"].le(daily["total_events"]).all()
assert segments["activity_segment"].notna().all()
assert retention["week_number"].ge(0).all()
assert retention["active_users"].le(retention["cohort_size"]).all()
assert temporal["day_of_week"].between(0, 6).all()
assert temporal["hour_of_day"].between(0, 23).all()
assert len(temporal) <= 168

w0 = retention.loc[retention["week_number"] == 0, "retention_rate"]
assert not w0.empty and np.isclose(w0, 100).all()

daily_event_total = int(daily["total_events"].sum())
segment_event_total = int(segments["total_events"].sum())
assert daily_event_total == segment_event_total

checks = pd.DataFrame({
    "check": [
        "Daily event decomposition", "Temporal event decomposition",
        "User event decomposition", "Unique visitor grain",
        "Retention W0 = 100%", "No negative retention week",
        "Temporal grain <= 168 buckets", "Daily/user event totals reconcile",
    ],
    "status": ["PASS"] * 8,
})
display(checks)

print(f"Total observed event rows: {daily_event_total:,}")
print(f"Sum of DAU across dates: {int(daily['dau'].sum()):,} user-days")
print("DAU summed over dates is not period-unique visitors.")


## 3. KPI overview

DAU — число уникальных посетителей за конкретный день. Сумма DAU за весь период означает user-days, а не число уникальных пользователей за весь период.

In [ ]:
# KPI overview
date_min = daily["event_date"].min()
date_max = daily["event_date"].max()

kpis = pd.Series({
    "Observation start (UTC)": date_min.date(),
    "Observation end (UTC)": date_max.date(),
    "Calendar days": (date_max - date_min).days + 1,
    "Total event rows": int(daily["total_events"].sum()),
    "Peak DAU": int(daily["dau"].max()),
    "Average DAU": round(daily["dau"].mean(), 2),
    "View event rows": int(daily["views"].sum()),
    "Add-to-cart event rows": int(daily["cart_events"].sum()),
    "Transaction event rows": int(daily["purchase_events"].sum()),
})
display(kpis.to_frame("value"))


In [ ]:
# Daily activity trends
daily_plot = daily.set_index("event_date").sort_index()
calendar = pd.date_range(daily_plot.index.min(), daily_plot.index.max(), freq="D")
daily_plot = daily_plot.reindex(calendar)
daily_plot.index.name = "event_date"

count_cols = ["total_events", "dau", "views", "cart_events", "purchase_events"]
daily_plot[count_cols] = daily_plot[count_cols].fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
daily_plot["dau"].plot(ax=ax, alpha=0.4, label="DAU")
daily_plot["dau"].rolling(7, min_periods=1).mean().plot(
    ax=ax, linewidth=2, label="7-day moving average"
)
ax.set(title="Daily Active Users", xlabel="Date (UTC)", ylabel="Unique visitors")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
daily_plot["total_events"].plot(ax=ax, alpha=0.35, label="Daily events")
daily_plot["total_events"].rolling(7, min_periods=1).mean().plot(
    ax=ax, linewidth=2, label="7-day moving average"
)
ax.set(title="Daily Event Volume", xlabel="Date (UTC)", ylabel="Event rows")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Conversion funnel

Витрина считает уникальных пользователей, у которых хотя бы раз встретился каждый тип события за весь период. Она не проверяет порядок событий: cart/purchase users не обязательно являются подмножеством view users. Это baseline user reach, а не последовательная воронка.

In [ ]:
# Whole-period baseline reach funnel
funnel_row = funnel.iloc[0]
funnel_counts = pd.Series({
    "View users": int(funnel_row["view_users"]),
    "Add-to-cart users": int(funnel_row["cart_users"]),
    "Purchase users": int(funnel_row["purchase_users"]),
})
funnel_rates = pd.Series({
    "View → Cart": float(funnel_row["view_to_cart_cr"]),
    "Cart → Purchase": float(funnel_row["cart_to_purchase_cr"]),
    "View → Purchase": float(funnel_row["view_to_purchase_cr"]),
})

display(funnel_counts.to_frame("unique visitors"))
display(funnel_rates.to_frame("rate, %"))

fig, ax = plt.subplots(figsize=(9, 5))
funnel_counts.plot(kind="bar", ax=ax)
ax.set(title="User Reach by Event Type — Full Period", xlabel="", ylabel="Unique visitors")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print("Do not interpret the stage differences as sequential funnel drop-off.")


### Daily same-day reach ratios

Дневные коэффициенты рассчитываются по уникальным пользователям, совершившим оба типа событий в пределах одного календарного дня. Это не проверка последовательности действий.

In [ ]:
# Daily conversion trend (same-day user reach ratios)
daily_plot["view_to_cart_cr"] = daily.set_index("event_date")["view_to_cart_cr"].reindex(calendar)
daily_plot["view_to_purchase_cr"] = daily.set_index("event_date")["view_to_purchase_cr"].reindex(calendar)

fig, ax = plt.subplots(figsize=(14, 5))
daily_plot["view_to_cart_cr"].rolling(7, min_periods=1).mean().plot(
    ax=ax, label="View → Cart (7-day average)"
)
daily_plot["view_to_purchase_cr"].rolling(7, min_periods=1).mean().plot(
    ax=ax, label="View → Purchase (7-day average)"
)
ax.set(title="Daily Same-Day User Reach Ratios", xlabel="Date (UTC)", ylabel="Rate, %")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Cohort retention

Когорта — неделя первого наблюдаемого события. W0 = 100% по определению. Витрина хранит только недели с наблюдаемой активностью; отсутствующая строка не равна автоматически нулю. Последние когорты имеют неполный период наблюдения (right-censoring).

In [ ]:
# Weekly cohort retention heatmap
retention_pivot = retention.pivot(
    index="cohort_week", columns="week_number", values="retention_rate"
).sort_index()

fig, ax = plt.subplots(figsize=(14, max(6, len(retention_pivot) * 0.35)))
sns.heatmap(
    retention_pivot, annot=True, fmt=".1f", cmap="Blues",
    linewidths=0.3, cbar_kws={"label": "Retention, %"}, ax=ax
)
ax.set(title="Weekly Cohort Retention", xlabel="Weeks since first observed activity", ylabel="Cohort week (UTC)")
plt.tight_layout()
plt.show()

display(retention_pivot.tail())


## 6. User activity segments

Low = 1 событие, Medium = 2–3, High = 4+. Сегменты измеряют интенсивность событий, а не ценность клиента. Purchase rate — доля посетителей сегмента хотя бы с одним transaction event.

In [ ]:
# Segment profile
segment_order = ["Low Activity", "Medium Activity", "High Activity"]
segment_summary = (
    segments.groupby("activity_segment")
    .agg(
        users=("visitorid", "nunique"),
        avg_events=("total_events", "mean"),
        avg_views=("views", "mean"),
        avg_add_to_carts=("add_to_carts", "mean"),
        avg_transaction_event_rows=("purchases", "mean"),
        purchase_users=("purchases", lambda values: int((values > 0).sum())),
    )
    .reindex(segment_order)
)
segment_summary["user_share_pct"] = segment_summary["users"] / segment_summary["users"].sum() * 100
segment_summary["purchase_rate_pct"] = segment_summary["purchase_users"] / segment_summary["users"] * 100
display(segment_summary.round(2))


In [ ]:
# Segment size and purchase rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
segment_summary["users"].plot(kind="bar", ax=axes[0])
axes[0].set(title="Users by Activity Segment", xlabel="", ylabel="Unique visitors")
axes[0].tick_params(axis="x", rotation=20)

segment_summary["purchase_rate_pct"].plot(kind="bar", ax=axes[1])
axes[1].set(title="Visitors with a Purchase Event", xlabel="", ylabel="Purchase rate, %")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

print(
    "Caution: transaction events contribute to total_events, which defines the segments. "
    "The association between activity segment and purchase rate is descriptive and partly "
    "mechanically coupled; it is not causal."
)


## 7. Temporal activity

Сравниваем weekday/hour buckets. Чтобы учесть, что в календарном диапазоне может быть разное число понедельников, вторников и т. д., нормируем объём событий на число календарных вхождений соответствующего дня недели. Все часы — UTC.

In [ ]:
# Normalize by calendar exposure
start_date = date_min.normalize()
end_date = date_max.normalize()
calendar_days = pd.date_range(start_date, end_date, freq="D")

# Convert pandas weekday (Mon=0) to PostgreSQL DOW (Sun=0).
weekday_exposure = calendar_days.dayofweek.value_counts().sort_index()
weekday_exposure.index = (weekday_exposure.index + 1) % 7
weekday_exposure = weekday_exposure.sort_index()

full_grid = pd.MultiIndex.from_product(
    [range(7), range(24)], names=["day_of_week", "hour_of_day"]
).to_frame(index=False)

temporal_full = full_grid.merge(
    temporal, on=["day_of_week", "hour_of_day"], how="left"
)
metric_cols = ["total_events", "unique_users", "views", "add_to_carts", "purchases"]
temporal_full[metric_cols] = temporal_full[metric_cols].fillna(0)
temporal_full["calendar_occurrences"] = temporal_full["day_of_week"].map(weekday_exposure)
temporal_full["events_per_occurrence"] = (
    temporal_full["total_events"] / temporal_full["calendar_occurrences"]
)
temporal_full["purchases_per_occurrence"] = (
    temporal_full["purchases"] / temporal_full["calendar_occurrences"]
)

weekday_labels = {0:"Sun", 1:"Mon", 2:"Tue", 3:"Wed", 4:"Thu", 5:"Fri", 6:"Sat"}
temporal_full["weekday"] = temporal_full["day_of_week"].map(weekday_labels)


In [ ]:
# Average events by weekday and hour
weekday_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
events_heatmap = temporal_full.pivot(
    index="weekday", columns="hour_of_day", values="events_per_occurrence"
).reindex(weekday_order)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    events_heatmap, cmap="YlGnBu", ax=ax,
    cbar_kws={"label": "Average events per calendar occurrence"}
)
ax.set(title="Average Event Activity by Weekday and Hour (UTC)", xlabel="Hour (UTC)", ylabel="")
plt.tight_layout()
plt.show()

purchase_heatmap = temporal_full.pivot(
    index="weekday", columns="hour_of_day", values="purchases_per_occurrence"
).reindex(weekday_order)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    purchase_heatmap, cmap="YlOrRd", ax=ax,
    cbar_kws={"label": "Average transaction event rows per occurrence"}
)
ax.set(title="Average Purchase Event Activity by Weekday and Hour (UTC)", xlabel="Hour (UTC)", ylabel="")
plt.tight_layout()
plt.show()


## 8. Descriptive findings

Автоматически выводим несколько фактов. Они описывают наблюдаемые данные, но не доказывают причинно-следственные связи.

In [ ]:
# Data-backed descriptive observations
peak_dau = daily.loc[daily["dau"].idxmax()]
peak_events = daily.loc[daily["total_events"].idxmax()]
best_segment = segment_summary["purchase_rate_pct"].idxmax()

print(f"Peak DAU: {int(peak_dau['dau']):,} on {peak_dau['event_date'].date()}.")
print(f"Peak events: {int(peak_events['total_events']):,} on {peak_events['event_date'].date()}.")
print(
    f"Highest purchase rate among activity segments: {best_segment} "
    f"({segment_summary.loc[best_segment, 'purchase_rate_pct']:.2f}%)."
)

top_time_buckets = temporal_full.nlargest(
    5, "events_per_occurrence"
)[["weekday", "hour_of_day", "events_per_occurrence"]]
display(top_time_buckets)


## 9. Optional exports

Экспортируем только агрегированные результаты для локального просмотра. Для DataLens предпочтительнее подключаться непосредственно к PostgreSQL Data Marts. Папка `data/processed/` исключена из Git.

In [ ]:
# Optional local CSV exports
EXPORT_DIR = PROJECT_ROOT / "data" / "processed" / "product_analytics"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

daily.to_csv(EXPORT_DIR / "daily_kpis.csv", index=False)
funnel.to_csv(EXPORT_DIR / "conversion_funnel.csv", index=False)
retention.to_csv(EXPORT_DIR / "weekly_retention.csv", index=False)
segment_summary.reset_index().to_csv(EXPORT_DIR / "user_segments_summary.csv", index=False)
temporal_full.to_csv(EXPORT_DIR / "temporal_activity_normalized.csv", index=False)

print(f"Exports saved to: {EXPORT_DIR}")


## 10. Product Analyst conclusion template

Для итогового портфолио сформулируй каждый вывод по схеме:

1. **Observation:** что видно в данных? Укажи метрику, значение и период.
2. **Interpretation:** какие продуктовые объяснения возможны? Отделяй гипотезу от факта.
3. **Limitation:** какие ограничения данных влияют на вывод?
4. **Next step:** какой дополнительный анализ или эксперимент проверит объяснение?

Не утверждай, что день недели, время или активность вызывают покупку: для причинного вывода нужен эксперимент.

**Остаётся отдельными этапами:** интерактивный DataLens Dashboard и математико-статистический блок (confidence intervals, tests, MDE, power, A/B testing). Строгая последовательная/session-level воронка потребует отдельной session model.